# 01 EDA (member A)

Goal: understand the data and find the problems to fix. Every figure is saved to `reports/figures/` for the report and slides.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # so the notebook can import src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import FIGURE_DIR, TARGET
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

In [2]:
from src.data import load_data
train, test = load_data()
print(train.shape, test.shape)
train.head()

(1460, 81) (1459, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## 1. Target: SalePrice
Right-skewed -> close to normal after log (iter1 uses log; control: iter1_no_log).

In [3]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(train[TARGET], kde=True, ax=ax[0]).set_title(f"SalePrice (skew={train[TARGET].skew():.2f})")
sns.histplot(np.log1p(train[TARGET]), kde=True, ax=ax[1]).set_title(f"log1p(SalePrice) (skew={np.log1p(train[TARGET]).skew():.2f})")
plt.tight_layout(); plt.savefig(FIGURE_DIR / "target_distribution.png", dpi=150)

## 2. Features most correlated with price

In [4]:
corr = train.select_dtypes("number").corr()[TARGET].drop(TARGET).sort_values(ascending=False)
top = corr.head(10)
print(top)
plt.figure(figsize=(8, 4)); sns.barplot(x=top.values, y=top.index)
plt.title("Top 10 correlation with SalePrice"); plt.tight_layout()
plt.savefig(FIGURE_DIR / "top_correlations.png", dpi=150)

OverallQual     0.790982
GrLivArea       0.708624
GarageCars      0.640409
GarageArea      0.623431
TotalBsmtSF     0.613581
1stFlrSF        0.605852
FullBath        0.560664
TotRmsAbvGrd    0.533723
YearBuilt       0.522897
YearRemodAdd    0.507101
Name: SalePrice, dtype: float64


In [5]:
cols = [TARGET] + list(top.index[:6])
plt.figure(figsize=(9, 7)); sns.heatmap(train[cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.tight_layout(); plt.savefig(FIGURE_DIR / "corr_heatmap.png", dpi=150)

## 3. Outliers
Bottom right: huge living area but low price (removed in iter2).

In [6]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=train, x="GrLivArea", y=TARGET)
out = train[(train.GrLivArea > 4000) & (train[TARGET] < 300000)]
plt.scatter(out.GrLivArea, out[TARGET], color="red", s=80, label="outliers"); plt.legend()
plt.tight_layout(); plt.savefig(FIGURE_DIR / "outliers.png", dpi=150)
out[["Id", "GrLivArea", TARGET]]

,Id,GrLivArea,SalePrice
523,524,4676,184750
1298,1299,5642,160000


## 4. Missing values
Note: many NAs mean "no such facility", see `data_description.txt` (handled by fill_missing in iter2).

In [7]:
full = pd.concat([train.drop(columns=TARGET), test])
missing = (full.isna().mean() * 100).sort_values(ascending=False)
missing = missing[missing > 0]
print(f"{len(missing)} columns have missing values")
shown = missing[missing >= 0.5]  # the rest are only a handful of rows each
plt.figure(figsize=(8, 6)); ax = sns.barplot(x=shown.values, y=shown.index)
ax.bar_label(ax.containers[0], fmt="%.1f%%", padding=3, fontsize=8)
ax.set(xlabel="% missing (train + test)", ylabel="", title=f"Columns with >= 0.5% missing ({len(shown)} of {len(missing)})")
plt.tight_layout(); plt.savefig(FIGURE_DIR / "missing_values.png", dpi=150)
missing

34 columns have missing values


PoolQC          99.657417
MiscFeature     96.402878
Alley           93.216855
Fence           80.438506
MasVnrType      60.500171
FireplaceQu     48.646797
LotFrontage     16.649538
GarageYrBlt      5.447071
GarageCond       5.447071
GarageFinish     5.447071
GarageQual       5.447071
GarageType       5.378554
BsmtCond         2.809181
BsmtExposure     2.809181
BsmtQual         2.774923
BsmtFinType2     2.740665
BsmtFinType1     2.706406
MasVnrArea       0.787941
MSZoning         0.137033
BsmtHalfBath     0.068517
Utilities        0.068517
Functional       0.068517
BsmtFullBath     0.068517
BsmtFinSF1       0.034258
Exterior1st      0.034258
TotalBsmtSF      0.034258
BsmtUnfSF        0.034258
BsmtFinSF2       0.034258
SaleType         0.034258
KitchenQual      0.034258
GarageCars       0.034258
GarageArea       0.034258
Electrical       0.034258
Exterior2nd      0.034258
dtype: float64

## 5. Categorical features: e.g. Neighborhood / OverallQual

In [8]:
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
order = train.groupby("Neighborhood")[TARGET].median().sort_values().index
sns.boxplot(data=train, x="Neighborhood", y=TARGET, order=order, ax=ax[0]); ax[0].tick_params(axis="x", rotation=90)
sns.boxplot(data=train, x="OverallQual", y=TARGET, ax=ax[1])
plt.tight_layout(); plt.savefig(FIGURE_DIR / "categorical_vs_price.png", dpi=150)

## Findings (for report section 2)

- **Data size:** 1,460 training houses and 1,459 test houses, 79 features (36 numeric, 43 categorical).
- **Target is right-skewed:** SalePrice skew = 1.88 (median $163,000, mean $180,921). After `log1p` the skew drops to 0.12, close to normal. The competition metric is also measured on log prices, so the baseline already trains on log(SalePrice).
- **Quality and size drive price:** the strongest correlations are OverallQual (0.79), GrLivArea (0.71), GarageCars (0.64), GarageArea (0.62) and TotalBsmtSF (0.61). Median price rises steadily from $50k at OverallQual 1 to $432k at OverallQual 10.
- **Location matters:** median price by Neighborhood ranges from $88k (MeadowV) to $315k (NridgHt), so Neighborhood is worth keeping as a category.
- **Two clear outliers:** Id 524 and 1299 have GrLivArea > 4,000 sq ft but sold for under $200k (both Partial sales in Edwards). They break the size-price trend. -> removed in Stage 2.
- **Missing values are mostly "no facility":** 34 columns have NAs. PoolQC (99.7%), MiscFeature (96.4%), Alley (93.2%), Fence (80.4%), FireplaceQu (48.6%) and the Garage*/Bsmt* columns are NA because the house has no pool/alley/fence/fireplace/garage/basement (see `data_description.txt`). They should be filled with "None" (or 0 for areas/counts), not with the median. LotFrontage (16.6%) is truly missing -> fill by Neighborhood median. -> Stage 2.

## Stage 1 results

| Version | CV RMSE (log) | Per-fold | Kaggle |
|---|---|---|---|
| `iter1_no_log` (control, raw SalePrice) | 0.15389 ± 0.01588 | 0.157, 0.149, **0.180**, 0.153, 0.130 | not submitted |
| `iter1_baseline` (log SalePrice) | 0.14644 ± 0.03923 | 0.136, 0.129, **0.224**, 0.127, 0.116 | **0.13241** |

- The log transform improves 4 of 5 folds by about 0.02.
- The bold fold is the one that contains the 2 outliers. With log it jumps to 0.224 (vs ~0.13 elsewhere), which is why the std is high. This is the main motivation for removing outliers in Stage 2.


# Stage 2: data cleaning

Cleaning code: `remove_outliers` in `src/data.py` and `fill_missing` in `src/features.py`.

Rules:
1. **Outliers:** drop the 2 houses with GrLivArea > 4,000 and SalePrice < $300k (Id 524, 1299).
2. **Facility present but a field is NA** (e.g. MasVnrArea > 0 but MasVnrType NA): truly missing -> most common value among houses that have the facility.
3. **NA = no facility** (pool, alley, fence, fireplace, garage, basement, veneer, misc): -> `"None"` for categories, `0` for areas/counts.
4. **GarageYrBlt:** fix the test-set typo 2207 -> remodel year 2007; houses without a garage get the house's YearBuilt instead of 0.
5. **LotFrontage:** median of the same Neighborhood.
6. **Anything left** (a few rows in MSZoning, Utilities, KitchenQual, ...): mode / median.

In [9]:
from src.data import remove_outliers
from src.features import fix_types, fill_missing

full = pd.concat([train.drop(columns=TARGET), test], ignore_index=True)
cleaned = fill_missing(fix_types(full))
print("NA before:", int(full.isna().sum().sum()), "-> after:", int(cleaned.isna().sum().sum()))

# Rows where the facility exists but a field was NA (rule 2)
check = {"MasVnrType": full.MasVnrType.isna() & (full.MasVnrArea > 0),
         "PoolQC": full.PoolQC.isna() & (full.PoolArea > 0),
         "BsmtQual": full.BsmtQual.isna() & (full.TotalBsmtSF > 0),
         "GarageFinish": full.GarageFinish.isna() & (full.GarageArea > 0)}
for col, mask in check.items():
    print(f"{col:<13} {mask.sum()} rows -> {cleaned.loc[mask, col].unique().tolist()}")
print("GarageYrBlt range:", cleaned.GarageYrBlt.min(), "-", cleaned.GarageYrBlt.max())

NA before: 15707 -> after: 0
MasVnrType    8 rows -> ['BrkFace']
PoolQC        3 rows -> ['Ex']
BsmtQual      2 rows -> ['TA']
GarageFinish  1 rows -> ['Unf']
GarageYrBlt range: 1872.0 - 2010.0


In [10]:
clean_train = remove_outliers(train)
fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
sns.scatterplot(data=train, x="GrLivArea", y=TARGET, ax=ax[0]).set_title(f"Before ({len(train)} houses)")
sns.scatterplot(data=clean_train, x="GrLivArea", y=TARGET, ax=ax[1]).set_title(f"After removing outliers ({len(clean_train)} houses)")
plt.tight_layout(); plt.savefig(FIGURE_DIR / "outliers_before_after.png", dpi=150)

## Stage 2 results

| Version | Change vs iter1 | CV RMSE (log) | Per-fold | Kaggle |
|---|---|---|---|---|
| `iter1_baseline` | - | 0.14644 ± 0.03923 | 0.136, 0.129, **0.224**, 0.127, 0.116 | 0.13241 |
| `iter2b_missing_only` | missing values only | 0.14615 ± 0.03875 | 0.138, 0.129, **0.222**, 0.128, 0.113 | not submitted |
| `iter2a_outliers_only` | outliers only | 0.11465 ± 0.00812 | 0.122, 0.110, 0.116, 0.123, 0.101 | not submitted |
| `iter2_cleaned` | both | 0.11477 ± 0.00884 | 0.124, 0.111, 0.117, 0.123, 0.100 | 0.13281 |

### The CV gain did not show up on Kaggle

CV improved from 0.146 to 0.115, but the Kaggle score stayed flat (0.13241 -> 0.13281). The reason is that the two CV numbers are not measured on the same houses: once the outliers are removed from the data, they also disappear from the validation folds, so iter2 is no longer scored on the 2 hardest houses.

**Fair comparison:** same 5 folds over all 1,460 houses. Outliers are removed only from the *training* part, and each model is scored on the same houses:

| Version | Scored on all 1,460 houses | Scored on the 1,458 normal houses |
|---|---|---|
| `iter1_baseline` | 0.14644 | 0.12309 |
| `iter2b_missing_only` | 0.14615 | 0.12324 |
| `iter2a_outliers_only` | 0.13722 | 0.11387 |
| `iter2_cleaned` | 0.13696 | 0.11405 |

- **Removing the outliers really does help:** on the same normal houses the error drops by ~0.009 (0.1231 -> 0.1140), because the 2 outliers no longer pull the fitted line down for large houses. The honest CV gain is ~0.009, not the ~0.032 the naive CV suggested.
- **Kaggle did not confirm it** (+0.0004, within noise). The public leaderboard scores only about half of the test set (~730 houses), so a few unusual houses can move the score by this much. One candidate is test Id 2550: a 5,095 sq ft Partial sale in Edwards, just like the 2 removed outliers. A model trained without the outliers will predict a high price for it, and if it actually sold cheaply, that single house can cost a few thousandths of RMSE. We cannot check this, because the test labels are hidden.
- **Missing-value handling barely changes the score for Ridge** (0.1231 -> 0.1232 on normal houses). Its value is correctness: "no garage" and "unknown garage" are different things, and Stage 3 features (e.g. garage/house age) need sensible values instead of 0 or NaN.
- The refined rules (2) and (4) changed CV by only +0.0002 compared with a plain "NA -> None/0" fill, which is within noise. They are kept because they are more correct and avoid a fake "garage built in year 0".
- **Lesson:** CV is only comparable when every version is scored on the same rows. Check big CV jumps against Kaggle before trusting them.

**Decision:** keep the outlier removal for Stage 3 and 4. On a like-for-like comparison it improves predictions for normal houses, and the Kaggle change is within noise.

### Notes for Stage 3 (handoff to B)
- Id 2550 (test) has YrSold < YearBuilt and Id 1877 (test) has YearRemodAdd < YearBuilt, so `HouseAge` / `RemodAge` can be negative for these rows; consider clipping at 0.
- Compare iter3 against iter2 using `iter2_cleaned`'s CV (0.11477): both remove the same outliers, so they are scored on the same houses and the comparison is fair.